# FIM-Lifted UPDE vs Standard Kuramoto

The SCPN's Unified Phase Dynamics Equation is NOT standard Kuramoto.
It adds an Information-Geometric Lift: gradient flow on the Fisher
Information Metric manifold. This notebook asks:

**Does the FIM lift produce measurably different dynamics?**

If yes, the SCPN makes predictions that standard Kuramoto doesn't.
If no, the FIM lift is decorative and the SCPN reduces to Kuramoto.

**The FIM-lifted UPDE** (Paper 0, eq. 3.2):

dθ_i/dt = ω_i + Σ_j K_ij sin(θ_j - θ_i) + λ g^{ab} ∂_a F_i(θ)

where g^{ab} is the inverse Fisher metric on the statistical manifold
of phase distributions, and F_i is the free energy functional.

The extra term pulls phases toward the maximum of Fisher information —
the configuration where the system is most "informative" about its state.

In [ ]:
import numpy as np
from scipy.stats import pearsonr, ks_2samp

from scpn_quantum_control.bridge import build_knm_paper27, OMEGA_N_16

In [ ]:
def fisher_information_metric(phases):
    """Approximate FIM for circular phase distribution.
    
    For a von Mises distribution p(theta|mu,kappa) = exp(kappa*cos(theta-mu)) / (2*pi*I_0(kappa)),
    the Fisher information is F = kappa * I_1(kappa) / I_0(kappa).
    
    We estimate kappa from the sample concentration (R = |mean(exp(i*theta))|).
    For R close to 0: kappa ~ 2R. For R close to 1: kappa ~ 1/(1-R^2).
    """
    N = len(phases)
    z = np.exp(1j * phases)
    R = np.abs(np.mean(z))
    mu = np.angle(np.mean(z))
    
    # Approximate kappa from R (Mardia & Jupp, 2000)
    if R < 0.53:
        kappa = 2*R + R**3 + 5*R**5/6
    elif R < 0.85:
        kappa = -0.4 + 1.39*R + 0.43/(1-R)
    else:
        kappa = 1.0 / (R**3 - 4*R**2 + 3*R + 1e-10)
    kappa = max(0.01, min(kappa, 100.0))
    
    # FIM for von Mises: I(kappa) = kappa * A(kappa), A = I_1/I_0
    from scipy.special import i0, i1
    A = i1(kappa) / (i0(kappa) + 1e-10)
    fisher_info = kappa * A
    
    return fisher_info, kappa, mu, R


def fim_gradient(phases, i):
    """FIM gradient for oscillator i: direction that maximises Fisher info.
    
    dF/dtheta_i = d/dtheta_i [kappa * I_1(kappa)/I_0(kappa)]
    ~ (1/N) * sin(mu - theta_i) * dF/dR (chain rule through R)
    
    Maximising Fisher info = moving toward the mean phase (increases R,
    increases concentration, increases information).
    """
    N = len(phases)
    z = np.exp(1j * phases)
    mean_z = np.mean(z)
    mu = np.angle(mean_z)
    R = np.abs(mean_z)
    
    # Gradient: pull toward mean phase, scaled by how much it improves FIM
    # At low R: strong pull (system needs order). At high R: weak pull (already ordered).
    phase_diff = mu - phases[i]
    # Wrap to [-pi, pi]
    phase_diff = (phase_diff + np.pi) % (2*np.pi) - np.pi
    
    # FIM sensitivity: d(Fisher)/dR ~ 1/(1-R^2) for large R, ~2 for small R
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    
    return (1.0 / N) * np.sin(phase_diff) * min(sensitivity, 50.0)


def simulate_kuramoto(N, K, omega, T=50.0, dt=0.01, noise=0.05, 
                       fim_lambda=0.0, seed=42):
    """Standard Kuramoto (fim_lambda=0) or FIM-lifted UPDE (fim_lambda>0)."""
    rng = np.random.default_rng(seed)
    n_steps = int(T / dt)
    phases = rng.uniform(0, 2*np.pi, N)
    
    R_history = []
    FIM_history = []
    
    for t in range(n_steps):
        dphi = omega.copy()
        for i in range(N):
            # Standard Kuramoto coupling
            for j in range(N):
                dphi[i] += K[i,j] * np.sin(phases[j] - phases[i])
            # FIM lift
            if fim_lambda > 0:
                dphi[i] += fim_lambda * fim_gradient(phases, i)
        
        phases += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
        phases = phases % (2 * np.pi)
        
        if t % 10 == 0:
            R = float(np.abs(np.mean(np.exp(1j * phases))))
            fi, _, _, _ = fisher_information_metric(phases)
            R_history.append(R)
            FIM_history.append(fi)
    
    return np.array(R_history), np.array(FIM_history)


print('Functions defined.')

## 2. Head-to-Head: Standard Kuramoto vs FIM-Lifted UPDE

In [ ]:
N = 8
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

K_scales = [0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
fim_lambdas = [0.0, 0.5, 1.0, 2.0, 5.0]

print('=== STANDARD KURAMOTO vs FIM-LIFTED UPDE ===')
print(f'{"K_scale":>8}', end='')
for lam in fim_lambdas:
    print(f'  lam={lam:<4}', end='')
print('   Diff(0 vs 5)')
print('-' * 75)

all_results = []
for k_scale in K_scales:
    K = K_base * k_scale
    row = []
    for lam in fim_lambdas:
        R_hist, FIM_hist = simulate_kuramoto(N, K, omega, T=40.0, fim_lambda=lam)
        R_final = np.mean(R_hist[-50:])  # average over last portion
        row.append(R_final)
    
    diff = row[-1] - row[0]  # FIM lift effect
    print(f'{k_scale:8.1f}', end='')
    for r in row:
        print(f'{r:10.4f}', end='')
    print(f'{diff:+10.4f}', end='')
    if abs(diff) > 0.05:
        print('  ***')
    else:
        print()
    all_results.append({'K_scale': k_scale, 'R_by_lambda': row, 'diff': diff})

In [ ]:
# Where does FIM matter most?
print('\n=== WHERE DOES THE FIM LIFT CHANGE THE DYNAMICS? ===')
print()
diffs = [r['diff'] for r in all_results]
peak_idx = np.argmax(np.abs(diffs))

print(f'Maximum FIM effect: K_scale={K_scales[peak_idx]}, dR={diffs[peak_idx]:+.4f}')
print()

if K_scales[peak_idx] < 1.0:
    print('FIM lift matters most NEAR K_c (subcritical regime).')
    print('This means: the information-geometric term helps oscillators')
    print('synchronise when coupling alone is insufficient.')
    print('PHYSICAL INTERPRETATION: consciousness can self-bootstrap')
    print('via information geometry even with weak physical coupling.')
elif K_scales[peak_idx] > 2.0:
    print('FIM lift matters most far above K_c.')
    print('This means: information geometry modifies already-synchronised states.')
else:
    print('FIM lift matters most at intermediate coupling.')

# Is the effect statistically significant?
print('\n=== STATISTICAL TEST: Kuramoto vs UPDE ===')
K_test = K_base * K_scales[peak_idx]
R_standard = []
R_fim = []
for seed in range(20):
    R_s, _ = simulate_kuramoto(N, K_test, omega, T=30.0, fim_lambda=0.0, seed=seed)
    R_f, _ = simulate_kuramoto(N, K_test, omega, T=30.0, fim_lambda=5.0, seed=seed)
    R_standard.append(np.mean(R_s[-30:]))
    R_fim.append(np.mean(R_f[-30:]))

ks_stat, ks_p = ks_2samp(R_standard, R_fim)
mean_diff = np.mean(R_fim) - np.mean(R_standard)
print(f'  Standard Kuramoto: R = {np.mean(R_standard):.4f} +/- {np.std(R_standard):.4f}')
print(f'  FIM-lifted UPDE:   R = {np.mean(R_fim):.4f} +/- {np.std(R_fim):.4f}')
print(f'  Mean difference:   {mean_diff:+.4f}')
print(f'  KS test:           D={ks_stat:.4f}, p={ks_p:.4f}')
print(f'  Significant:       {"YES" if ks_p < 0.05 else "NO"}')

In [ ]:
# Does FIM lift change the CRITICAL COUPLING K_c?
print('\n=== DOES FIM LIFT SHIFT K_c? ===')
K_fine = np.linspace(0.01, 2.0, 30)
R_threshold = 0.5

for lam_test in [0.0, 2.0, 5.0]:
    Kc_est = None
    for k in K_fine:
        K = K_base * k
        R_hist, _ = simulate_kuramoto(N, K, omega, T=30.0, fim_lambda=lam_test)
        R_final = np.mean(R_hist[-30:])
        if R_final > R_threshold and Kc_est is None:
            Kc_est = k
            break
    print(f'  lambda={lam_test:.1f}: K_c ~ {Kc_est if Kc_est else ">2.0"}')

print()
print('If FIM lift LOWERS K_c: consciousness needs less physical coupling')
print('to synchronise when information geometry is active.')
print('This would be a NOVEL prediction unique to the SCPN.')

In [ ]:
import json
print('--- JSON RESULTS ---')
print(json.dumps({
    'n_oscillators': N,
    'peak_fim_effect_K': K_scales[peak_idx],
    'peak_fim_diff': round(diffs[peak_idx], 4),
    'ks_stat': round(ks_stat, 4),
    'ks_p': round(ks_p, 4),
    'mean_R_standard': round(float(np.mean(R_standard)), 4),
    'mean_R_fim': round(float(np.mean(R_fim)), 4),
    'significant': bool(ks_p < 0.05),
}, indent=2))